# 00 — Data Preparation
**Purpose:** Merge raw CSVs (water quality, landsat, terraclimate), clean, verify, visualize.

**Input:** Raw CSVs from Kaggle Dataset (`ey-water-quality-data/`)

**Output:** `train_base.parquet`, `val_base.parquet`

### Figures
1. Station Location Map — train vs validation across South Africa
2. Target Distributions — histograms + boxplots + skewness
3. Data Coverage — samples per station + temporal span
4. Missing Values — horizontal bar chart
5. Spatial Target Patterns — mean water quality per station on map

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# --- Plot style ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'figure.dpi': 120,
})

SEED = 42
np.random.seed(SEED)

# --- Paths (adjust for local vs Kaggle) ---
INPUT_DIR = '/kaggle/input/ey-water-quality-data'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

---
## 1. Load Raw Data

In [ ]:
# === Water quality (targets) ===
wq_train = pd.read_csv(f'{INPUT_DIR}/water_quality_training_dataset.csv')
print(f'Water Quality Training: {wq_train.shape}')
display(wq_train.head(3))
print('\nColumn dtypes:')
display(wq_train.dtypes)

In [ ]:
# === Feature datasets ===
landsat_train = pd.read_csv(f'{INPUT_DIR}/landsat_features_training.csv')
landsat_val   = pd.read_csv(f'{INPUT_DIR}/landsat_features_validation.csv')
terra_train   = pd.read_csv(f'{INPUT_DIR}/terraclimate_features_training.csv')
terra_val     = pd.read_csv(f'{INPUT_DIR}/terraclimate_features_validation.csv')
submission    = pd.read_csv(f'{INPUT_DIR}/submission_template.csv')

print(f'Landsat      — Train: {landsat_train.shape},  Val: {landsat_val.shape}')
print(f'TerraClimate — Train: {terra_train.shape},  Val: {terra_val.shape}')
print(f'Submission template:  {submission.shape}')

print('\nLandsat columns:', landsat_train.columns.tolist())
print('TerraClimate columns:', terra_train.columns.tolist())
print('Submission columns:', submission.columns.tolist())

---
## 2. Column Setup & Date Parsing

The data uses `Latitude + Longitude` as station identifiers (no explicit station ID).
We create a synthetic `station_id` from the coordinate pair.

In [ ]:
# Fixed column names (known from data inspection)
LAT_COL = 'Latitude'
LON_COL = 'Longitude'
DATE_COL = 'Sample Date'

# Target columns
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Create synthetic station_id from (lat, lon)
def add_station_id(df):
    df = df.copy()
    df['station_id'] = df[LAT_COL].round(6).astype(str) + '_' + df[LON_COL].round(6).astype(str)
    return df

wq_train = add_station_id(wq_train)
landsat_train = add_station_id(landsat_train)
landsat_val = add_station_id(landsat_val)
terra_train = add_station_id(terra_train)
terra_val = add_station_id(terra_val)
submission = add_station_id(submission)

STATION_COL = 'station_id'

print(f'Training stations:   {wq_train[STATION_COL].nunique()}')
print(f'Validation stations: {landsat_val[STATION_COL].nunique()}')

In [ ]:
# Parse dates (format: DD-MM-YYYY)
for df in [wq_train, landsat_train, landsat_val, terra_train, terra_val, submission]:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], format='%d-%m-%Y', dayfirst=True)

print(f'Date range (train): {wq_train[DATE_COL].min().date()} to {wq_train[DATE_COL].max().date()}')
print(f'Date range (val):   {landsat_val[DATE_COL].min().date()} to {landsat_val[DATE_COL].max().date()}')

---
## 3. Merge Datasets

In [ ]:
MERGE_KEYS = [LAT_COL, LON_COL, DATE_COL]

# --- Training ---
train_merged = wq_train.merge(landsat_train, on=MERGE_KEYS, how='left', suffixes=('', '_ls'))
print(f'After Landsat merge:      {train_merged.shape}')

train_merged = train_merged.merge(terra_train, on=MERGE_KEYS, how='left', suffixes=('', '_tc'))
print(f'After TerraClimate merge: {train_merged.shape}')

# Keep the station_id from the first df, drop duplicates
dup_cols = [c for c in train_merged.columns if c.endswith('_ls') or c.endswith('_tc')]
if dup_cols:
    print(f'Dropping duplicate merge cols: {dup_cols}')
    train_merged.drop(columns=dup_cols, inplace=True)

# --- Validation ---
val_merged = landsat_val.merge(terra_val, on=MERGE_KEYS, how='left', suffixes=('', '_tc'))
dup_cols = [c for c in val_merged.columns if c.endswith('_tc')]
if dup_cols:
    val_merged.drop(columns=dup_cols, inplace=True)
print(f'Validation merged:        {val_merged.shape}')

print(f'\n--- Final Columns ---')
print(f'Train: {train_merged.columns.tolist()}')
print(f'Val:   {val_merged.columns.tolist()}')

---
## 4. Data Quality Checks

In [ ]:
# Station overlap check (CRITICAL for spatial extrapolation)
train_stations = set(train_merged[STATION_COL].unique())
val_stations   = set(val_merged[STATION_COL].unique())
overlap = train_stations & val_stations

print(f'Training stations:   {len(train_stations)}')
print(f'Validation stations: {len(val_stations)}')
print(f'Overlapping:         {len(overlap)}')

if overlap:
    print(f'\n!!! WARNING: Station overlap detected!  {overlap}')
else:
    print('\n[OK] No station overlap — spatial extrapolation confirmed')

# Target stats
print(f'\n--- Target Summary ---')
for col in TARGET_COLS:
    print(f'  {col}: dtype={train_merged[col].dtype}, '
          f'nulls={train_merged[col].isnull().sum()} ({train_merged[col].isnull().mean()*100:.1f}%), '
          f'range=[{train_merged[col].min():.2f}, {train_merged[col].max():.2f}]')

# Duplicate rows check
n_dup = train_merged.duplicated(subset=MERGE_KEYS).sum()
print(f'\nDuplicate rows (same lat/lon/date): {n_dup}')

---
## FIGURE 1: Station Location Map (Train vs Validation)

In [ ]:
# Unique station locations
train_locs = train_merged.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
val_locs   = val_merged.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()

fig, ax = plt.subplots(figsize=(12, 10))

# Plot stations
ax.scatter(train_locs[LON_COL], train_locs[LAT_COL],
           c='#2196F3', s=80, alpha=0.8, edgecolors='white', linewidths=0.5,
           label=f'Training  ({len(train_locs)} stations)', zorder=3)
ax.scatter(val_locs[LON_COL], val_locs[LAT_COL],
           c='#FF5722', s=140, marker='*', edgecolors='white', linewidths=0.5,
           label=f'Validation ({len(val_locs)} stations)', zorder=4)

# Major South African cities for context
cities = {
    'Cape Town':      (-33.92, 18.42),
    'Johannesburg':   (-26.20, 28.04),
    'Durban':         (-29.86, 31.02),
    'Pretoria':       (-25.75, 28.19),
    'Port Elizabeth': (-33.96, 25.60),
    'Bloemfontein':   (-29.09, 26.16),
}
for city, (lat, lon) in cities.items():
    ax.plot(lon, lat, 'k^', markersize=7, zorder=5)
    ax.annotate(city, (lon, lat), textcoords='offset points', xytext=(6, 6),
               fontsize=8, color='#555555', fontstyle='italic')

ax.set_xlim(16, 33)
ax.set_ylim(-35, -22)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Station Locations: Training vs Validation\n'
             'Validation stations are at UNSEEN locations — this is a spatial extrapolation task')
ax.legend(fontsize=11, loc='lower left',
          frameon=True, fancybox=True, shadow=True, framealpha=0.9)
ax.grid(True, alpha=0.3)

# Info box
info_text = (f'Train: {len(train_stations)} stations, {len(train_merged):,} samples\n'
             f'Val:   {len(val_stations)} stations, {len(val_merged):,} samples\n'
             f'Overlap: {len(overlap)} stations')
ax.text(0.98, 0.02, info_text, transform=ax.transAxes, ha='right', va='bottom',
        fontsize=10, family='monospace',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9, edgecolor='gray'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_00_station_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## FIGURE 2: Target Distributions

In [ ]:
n_targets = len(TARGET_COLS)
fig, axes = plt.subplots(2, n_targets, figsize=(6 * n_targets, 10),
                         gridspec_kw={'height_ratios': [3, 1]})

colors = ['#2196F3', '#4CAF50', '#FF9800']

for i, col in enumerate(TARGET_COLS):
    vals = train_merged[col].dropna()
    skew = vals.skew()
    q1, q3 = vals.quantile([0.25, 0.75])
    iqr = q3 - q1
    n_outliers = ((vals < q1 - 1.5 * iqr) | (vals > q3 + 1.5 * iqr)).sum()

    # Row 1: Histogram
    ax = axes[0, i]
    ax.hist(vals, bins=50, color=colors[i], alpha=0.7, edgecolor='white')
    ax.axvline(vals.median(), color='red', linestyle='--', linewidth=2,
               label=f'median = {vals.median():.1f}')
    ax.axvline(vals.mean(), color='black', linestyle=':', linewidth=2,
               label=f'mean = {vals.mean():.1f}')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9, loc='upper right')
    ax.text(0.95, 0.75, f'skew = {skew:.2f}\nn = {len(vals):,}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

    # Row 2: Box plot
    ax2 = axes[1, i]
    bp = ax2.boxplot(vals, vert=False, patch_artist=True, widths=0.6)
    bp['boxes'][0].set(facecolor=colors[i], alpha=0.5)
    ax2.set_xlabel('Value')
    ax2.set_yticks([])
    ax2.text(0.02, 0.85, f'{n_outliers} outliers',
             transform=ax2.transAxes, fontsize=10, va='top',
             bbox=dict(facecolor='lightyellow', alpha=0.8))

fig.suptitle('Target Variable Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_00_target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## FIGURE 3: Data Coverage — Samples per Station + Year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Left: Samples per station (horizontal bar, sorted) ---
station_counts = train_merged[STATION_COL].value_counts().sort_values()

ax = axes[0]
y_pos = np.arange(len(station_counts))
ax.barh(y_pos, station_counts.values, color='#2196F3', alpha=0.7, height=0.8)
ax.axvline(station_counts.median(), color='red', linestyle='--', linewidth=1.5,
           label=f'median = {station_counts.median():.0f}')
ax.set_xlabel('Number of Samples')
ax.set_ylabel(f'Station index (of {len(station_counts)})')
ax.set_title(f'Samples per Station\nmin={station_counts.min()}, '
             f'max={station_counts.max()}, median={station_counts.median():.0f}')
ax.legend(fontsize=10)
# Only show a few y-tick labels to avoid clutter
tick_step = max(1, len(station_counts) // 10)
ax.set_yticks(y_pos[::tick_step])
ax.set_yticklabels(range(0, len(station_counts), tick_step))

# --- Right: Samples per year (cleaner than per-month) ---
ax2 = axes[1]
year_counts = train_merged[DATE_COL].dt.year.value_counts().sort_index()
ax2.bar(year_counts.index.astype(str), year_counts.values, color='#4CAF50', alpha=0.7,
        edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Year')
ax2.set_ylabel('Number of Samples')
ax2.set_title(f'Samples per Year\n{year_counts.index.min()} – {year_counts.index.max()}')
# Rotate labels so they don't overlap
ax2.tick_params(axis='x', rotation=45)
# Add count labels on bars
for bar_container in ax2.containers:
    ax2.bar_label(bar_container, fontsize=8, padding=2)

fig.suptitle('Data Coverage', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_00_data_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

---
## FIGURE 4: Missing Values

In [ ]:
null_pcts = train_merged.isnull().mean()
null_pcts_nonzero = null_pcts[null_pcts > 0].sort_values(ascending=True)  # ascending for barh

if len(null_pcts_nonzero) > 0:
    fig_height = max(4, len(null_pcts_nonzero) * 0.5 + 1)
    fig, ax = plt.subplots(figsize=(10, fig_height))

    bar_colors = ['#F44336' if v > 0.5 else '#FF9800' if v > 0.1 else '#4CAF50'
                  for v in null_pcts_nonzero.values]
    bars = ax.barh(null_pcts_nonzero.index, null_pcts_nonzero.values * 100,
                   color=bar_colors, edgecolor='white', linewidth=0.5, height=0.7)

    # Add percentage labels on each bar
    for bar, pct in zip(bars, null_pcts_nonzero.values):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{pct*100:.1f}%', va='center', fontsize=9)

    ax.set_xlabel('% Missing', fontsize=12)
    ax.set_title('Missing Values by Feature', fontsize=14, fontweight='bold')

    legend_elements = [
        mpatches.Patch(facecolor='#F44336', label='> 50% missing'),
        mpatches.Patch(facecolor='#FF9800', label='10 – 50% missing'),
        mpatches.Patch(facecolor='#4CAF50', label='< 10% missing'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    ax.margins(y=0.02)  # reduce extra whitespace

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_00_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[OK] No missing values in training data')

print(f'\nTotal columns: {len(train_merged.columns)}')
print(f'Columns with nulls: {len(null_pcts_nonzero)}')

---
## FIGURE 5: Spatial Distribution of Each Target

In [ ]:
# Mean target value per station
station_means = train_merged.groupby(STATION_COL).agg(
    **{LAT_COL: (LAT_COL, 'first'),
       LON_COL: (LON_COL, 'first'),
       **{t: (t, 'mean') for t in TARGET_COLS}}
).reset_index()

fig, axes = plt.subplots(1, n_targets, figsize=(7 * n_targets, 7))
if n_targets == 1:
    axes = [axes]

for i, col in enumerate(TARGET_COLS):
    ax = axes[i]

    # Training stations (colored by mean target value)
    sc = ax.scatter(
        station_means[LON_COL], station_means[LAT_COL],
        c=station_means[col], cmap='RdYlGn_r', s=60, alpha=0.85,
        edgecolors='gray', linewidths=0.3
    )
    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    cbar.ax.tick_params(labelsize=9)

    # Validation stations (unknown targets)
    ax.scatter(
        val_locs[LON_COL], val_locs[LAT_COL],
        c='black', marker='x', s=60, linewidths=1.5,
        label='Validation (to predict)', zorder=5
    )

    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(fontsize=9, loc='lower left')
    ax.set_xlim(16, 33)
    ax.set_ylim(-35, -22)
    ax.grid(True, alpha=0.2)

fig.suptitle('Mean Water Quality per Station  (train = colored dots, val = black x)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_00_spatial_targets.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Save Base Datasets

In [ ]:
train_merged.to_parquet(f'{OUTPUT_DIR}/train_base.parquet', index=False)
val_merged.to_parquet(f'{OUTPUT_DIR}/val_base.parquet', index=False)

print(f'[OK] Saved train_base.parquet: {train_merged.shape}')
print(f'[OK] Saved val_base.parquet:   {val_merged.shape}')

print(f'\n=== Key Info (for downstream notebooks) ===')
print(f'  STATION_COL = "{STATION_COL}"  (synthetic: lat_lon)')
print(f'  LAT_COL     = "{LAT_COL}"')
print(f'  LON_COL     = "{LON_COL}"')
print(f'  DATE_COL    = "{DATE_COL}"')
print(f'  TARGET_COLS = {TARGET_COLS}')
print(f'  MERGE_KEYS  = {MERGE_KEYS}')
print(f'\n  Feature columns (Landsat):      nir, green, swir16, swir22, NDMI, MNDWI')
print(f'  Feature columns (TerraClimate): pet')

---
## Summary

| Metric | Value |
|--------|-------|
| Training samples | 9,319 |
| Validation samples | 200 |
| Training stations (unique lat/lon) | ~162 |
| Validation stations | ~? |
| Station overlap | 0 (confirmed) |
| Landsat features | 6 (nir, green, swir16, swir22, NDMI, MNDWI) |
| TerraClimate features | 1 (pet) |
| Targets | 3 (Alkalinity, EC, DRP) |

### Figures Produced
1. `fig_00_station_map.png` — Where are the stations? Train vs val
2. `fig_00_target_distributions.png` — Histograms + boxplots + skewness
3. `fig_00_data_coverage.png` — Samples per station + yearly distribution
4. `fig_00_missing_values.png` — Which features have nulls?
5. `fig_00_spatial_targets.png` — Mean water quality per station on map